# HealthTrust Watch — Phase II: Exploratory NLP Analysis

**Course:** INFO 4360 — NLP for Real-World Problems  
**Application:** MyChart  
**Dataset:** Health App Reviews for Privacy & Trust (HARPT)  
**Analysis Path:** Path B — Understanding and Extraction

## 1. Data Loading and Initial Exploration

This notebook investigates user-reported concerns in MyChart healthcare application reviews. We begin by extracting MyChart reviews from HARPT and examining the dataset's structure before cleaning or text mining.

The analysis is descriptive rather than predictive. Existing trust-related annotations are not assumed to represent verified complaints.

In [1]:
import pandas as pd
from pathlib import Path

# Locate the HARPT dataset regardless of Jupyter's working directory
possible_paths = [
    Path("data/HARPT.csv"),
    Path("../data/HARPT.csv"),
    Path("HealthTrust-Watch/data/HARPT.csv"),
]

data_path = next(
    (path for path in possible_paths if path.is_file()),
    None
)

if data_path is None:
    raise FileNotFoundError(
        "HARPT.csv not found. Make sure the file is inside "
        "HealthTrust-Watch/data/"
    )

print("Loading dataset from:", data_path.resolve())

# Load the original HARPT dataset
df = pd.read_csv(data_path, low_memory=False)

# Inspect the complete dataset
print("\nTotal HARPT reviews:", len(df))
print("Number of columns:", len(df.columns))

print("\nColumn names:")
print(df.columns.tolist())

# Extract MyChart reviews only
mychart_df = df.loc[
    df["app_name"].astype(str).str.strip().eq("MyChart")
].copy()

print("\nMyChart reviews:", len(mychart_df))

display(mychart_df.head())

C:\Users\sahit\AppData\Roaming\Python\Python313\site-packages\pandas\core\arrays\masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Loading dataset from: C:\Users\sahit\OneDrive\Desktop\INFO4360\Project\HealthTrust-Watch\data\HARPT.csv

Total HARPT reviews: 480450
Number of columns: 16

Column names:
['reviewId', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'app_name', 'app_id', 'word_count', 'year', 'sentiment', 'App_Type', 'trust_dimension', 'confidence_score']

MyChart reviews: 93392


,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,app_name,app_id,word_count,year,sentiment,App_Type,trust_dimension,confidence_score
480,90d8b83b-2298-4bf6-bc3e-f624960ef387,very easy to use and follow up with my care,4,0,10.8.2,4/11/25 19:54,NaN,NaN,MyChart,epic.mychart.android,10,2025,NaN,Patient Portal,data control,0.443538
481,ee61cbec-58f5-4e6a-ab67-d3e1bb8415e7,"Best information, since the wheel.",5,0,11.3.2,4/11/25 19:39,NaN,NaN,MyChart,epic.mychart.android,5,2025,NaN,Patient Portal,data quality,0.982839
482,2617f3cf-1cda-40c9-83c7-412c788f02cb,i found everything i was looking for plus some...,5,0,11.3.1,4/11/25 19:39,NaN,NaN,MyChart,epic.mychart.android,13,2025,NaN,Patient Portal,reliability,0.512631
483,9c1abef6-d1f2-4842-8d5f-4145d5b13a54,MyChart is an incredibly convenient app for ma...,5,0,NaN,4/11/25 19:31,NaN,NaN,MyChart,epic.mychart.android,51,2025,NaN,Patient Portal,data control,0.973237
484,5e606ce4-a11d-4cef-a506-135c95c26862,"it's very helpful, all involved are able to ac...",5,0,11.3.2,4/11/25 19:23,NaN,NaN,MyChart,epic.mychart.android,16,2025,NaN,Patient Portal,data control,0.989216


## 2. Data Quality Assessment

Before performing NLP analysis, we evaluate the completeness and consistency of the extracted MyChart reviews.

This assessment examines missing values, duplicate records, review-text quality, and the availability of important metadata.

These checks help determine which observations are suitable for subsequent text mining and which limitations must be considered when interpreting results.

In [2]:
# Data quality assessment for MyChart reviews

print("Dataset shape:", mychart_df.shape)

# Missing values
missing_summary = pd.DataFrame({
    "Missing Count": mychart_df.isna().sum(),
    "Missing Percentage": (
        mychart_df.isna().mean() * 100
    ).round(2)
})

display(missing_summary)

# Duplicate review IDs
duplicate_ids = mychart_df["reviewId"].duplicated().sum()

# Duplicate review text
duplicate_text = mychart_df["content"].duplicated().sum()

print("\nDuplicate review IDs:", duplicate_ids)
print("Duplicate review texts:", duplicate_text)

# Empty or whitespace-only reviews
empty_reviews = (
    mychart_df["content"].fillna("").astype(str).str.strip() == ""
).sum()

print("Empty review texts:", empty_reviews)

# Data types
print("\nColumn data types:")
display(mychart_df.dtypes.to_frame("Data Type"))

Dataset shape: (93392, 16)


,Missing Count,Missing Percentage
reviewId,0,0.00
content,46,0.05
score,0,0.00
thumbsUpCount,0,0.00
reviewCreatedVersion,8050,8.62
at,0,0.00
replyContent,9832,10.53
repliedAt,9832,10.53
app_name,0,0.00
app_id,0,0.00



Duplicate review IDs: 0
Duplicate review texts: 12699
Empty review texts: 46

Column data types:


,Data Type
reviewId,str
content,str
score,int64
thumbsUpCount,int64
reviewCreatedVersion,str
at,str
replyContent,str
repliedAt,str
app_name,str
app_id,str


### Data Quality Findings

The extracted MyChart dataset contains 93,392 reviews.

The initial missing-value assessment shows that:

- Only 46 reviews (0.05%) have missing review text.
- Star ratings and review dates have no missing values.
- Application-version information is missing for 8,050 reviews (8.62%).
- Developer reply information is missing for 9,832 reviews (10.53%).
- The existing sentiment column is entirely empty and will not be used as an analytical input.

These results indicate that the dataset has sufficient text and rating information for exploratory NLP analysis.

Reviews without usable text will be excluded from text analysis. Missing application-version information will be retained as a limitation rather than used as a reason to discard otherwise valid reviews.

The existing trust-dimension labels will not be treated as verified evidence of user complaints.

In [3]:
# Create a working copy without modifying the original dataset
mychart_clean = mychart_df.copy()

# Remove missing or whitespace-only review text
mychart_clean = mychart_clean.dropna(subset=["content"])

mychart_clean["content"] = (
    mychart_clean["content"].astype(str).str.strip()
)

mychart_clean = mychart_clean[
    mychart_clean["content"].ne("")
].copy()

# Parse review dates
mychart_clean["at"] = pd.to_datetime(
    mychart_clean["at"],
    format="%m/%d/%y %H:%M",
    errors="coerce"
)

# Convert ratings to numeric values
mychart_clean["score"] = pd.to_numeric(
    mychart_clean["score"],
    errors="coerce"
)

# Exclude records with invalid ratings or dates
mychart_clean = mychart_clean.dropna(subset=["score", "at"])

# Remove duplicate review IDs, retaining the first occurrence
mychart_clean = mychart_clean.drop_duplicates(
    subset=["reviewId"],
    keep="first"
)

# Calculate review length from the cleaned text
mychart_clean["review_length_words"] = (
    mychart_clean["content"].str.split().str.len()
)

mychart_clean["review_length_chars"] = (
    mychart_clean["content"].str.len()
)

print("Original MyChart reviews:", len(mychart_df))
print("Cleaned MyChart reviews:", len(mychart_clean))
print("Reviews excluded:", len(mychart_df) - len(mychart_clean))

print("\nDate range:")
print("Earliest:", mychart_clean["at"].min())
print("Latest:", mychart_clean["at"].max())

display(mychart_clean.head())

Original MyChart reviews: 93392
Cleaned MyChart reviews: 93346
Reviews excluded: 46

Date range:
Earliest: 2011-11-01 14:21:00
Latest: 2025-04-12 01:29:00


,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,app_name,app_id,word_count,year,sentiment,App_Type,trust_dimension,confidence_score,review_length_words,review_length_chars
480,90d8b83b-2298-4bf6-bc3e-f624960ef387,very easy to use and follow up with my care,4,0,10.8.2,2025-04-11 19:54:00,NaN,NaN,MyChart,epic.mychart.android,10,2025,NaN,Patient Portal,data control,0.443538,10,43
481,ee61cbec-58f5-4e6a-ab67-d3e1bb8415e7,"Best information, since the wheel.",5,0,11.3.2,2025-04-11 19:39:00,NaN,NaN,MyChart,epic.mychart.android,5,2025,NaN,Patient Portal,data quality,0.982839,5,34
482,2617f3cf-1cda-40c9-83c7-412c788f02cb,i found everything i was looking for plus some...,5,0,11.3.1,2025-04-11 19:39:00,NaN,NaN,MyChart,epic.mychart.android,13,2025,NaN,Patient Portal,reliability,0.512631,13,66
483,9c1abef6-d1f2-4842-8d5f-4145d5b13a54,MyChart is an incredibly convenient app for ma...,5,0,NaN,2025-04-11 19:31:00,NaN,NaN,MyChart,epic.mychart.android,51,2025,NaN,Patient Portal,data control,0.973237,51,332
484,5e606ce4-a11d-4cef-a506-135c95c26862,"it's very helpful, all involved are able to ac...",5,0,11.3.2,2025-04-11 19:23:00,NaN,NaN,MyChart,epic.mychart.android,16,2025,NaN,Patient Portal,data control,0.989216,16,92


### Data Cleaning Results and Interpretation

The original MyChart subset contained **93,392 reviews**. After excluding records with missing or unusable review text and applying data-quality checks, **93,346 reviews (99.95%)** remained, representing a reduction of only 46 records (0.05%).

The cleaned dataset spans **November 1, 2011, to April 12, 2025**, allowing historical review patterns to be explored.

The cleaning process preserves original review ratings and available application-version information. Reviews with missing version metadata remain in the general analysis because they can still contribute to keyword, sentiment, and topic exploration.

Duplicate review IDs are checked separately from repeated review text because identical comments may be submitted by different users. The specific reasons for excluded records will be verified before finalizing the audit.

Overall, the dataset provides a substantial foundation for exploratory NLP analysis. However, findings will describe patterns within the HARPT MyChart sample rather than assume that the sample represents all MyChart users.

In [4]:
# Verify the reasons for excluded reviews

missing_text = mychart_df["content"].isna().sum()

empty_text = (
    mychart_df["content"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
).sum()

duplicate_ids = mychart_df["reviewId"].duplicated().sum()

duplicate_texts = (
    mychart_df["content"]
    .dropna()
    .duplicated()
    .sum()
)

print("Missing review text:", missing_text)
print("Missing or blank review text:", empty_text)
print("Duplicate review IDs:", duplicate_ids)
print("Repeated review texts:", duplicate_texts)
print("Total excluded during cleaning:", len(mychart_df) - len(mychart_clean))

Missing review text: 46
Missing or blank review text: 46
Duplicate review IDs: 0
Repeated review texts: 12654
Total excluded during cleaning: 46


### Final Data Quality Assessment

The original MyChart dataset contained 93,392 reviews. Following the data-quality assessment, 46 records (0.05%) were excluded because their review text was missing. No additional blank-text records or duplicate review IDs were identified.

The final cleaned dataset contains **93,346 reviews**, representing a retention rate of **99.95%**.

Although 12,654 review texts were repeated, these records had distinct review IDs. They were retained because identical text does not necessarily indicate duplicate submissions. However, repeated generic comments may influence word-frequency analysis and will be considered when interpreting results.

The dataset covers November 2011 through April 2025. Application-version information is missing for some reviews, so version-based comparisons will require additional checks.

Overall, the dataset is suitable for exploratory NLP analysis, subject to limitations involving sample representativeness, annotation reliability, and repeated review text.